# Reproduce the iMSE benchmark in Colab

1. Select a GPU in **Runtime → Change runtime type**.
2. For ImageNet-1K downloads, accept the [ImageNet-1K access terms](https://huggingface.co/datasets/ILSVRC/imagenet-1k) and enable a read token named `HF_TOKEN` in **Colab Secrets**.
3. Choose the dataset and run the cells. Set `USE_DRIVE=True` to reuse saved features and results across sessions.

For a first run, start with CIFAR-10. Full ImageNet-1K needs substantial storage and compute. Vim-Base also requires a CUDA toolkit compatible with the runtime's PyTorch.

In [ ]:
# @title Benchmark settings
from pathlib import Path
import os
import subprocess
import sys

DATASET = "CIFAR10"  # @param ["CIFAR10", "CIFAR100", "ImageNet1K"]
BACKBONE = ""  # @param {type:"string"}
STAGE = "all"  # @param ["all", "download", "extract", "evaluate"]
USE_DRIVE = False  # @param {type:"boolean"}
DEVICE = "cuda"
REPO = Path("/content/iMSE")
WORK = Path("/content/iMSE-work")

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/iMSE-work")

In [ ]:
# @title Install the benchmark
from google.colab import userdata

if not os.environ.get("HF_TOKEN"):
    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
    if token:
        os.environ["HF_TOKEN"] = token
    del token

if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/erantascesme/iMSE.git", str(REPO)], check=True)
if not (REPO / "imse_benchmark").is_dir():
    raise RuntimeError(f"Update the iMSE checkout at {REPO} before running this notebook.")

needs_vim = (DATASET == "ImageNet1K" and BACKBONE in {"", "Vim-Base"}
             and STAGE in {"all", "extract"})
package = f"{REPO}[vim]" if needs_vim else str(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", package], check=True)
if STAGE != "download":
    subprocess.run([sys.executable, "-c",
                    "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime in Colab'"],
                   check=True)

In [ ]:
# @title Run the benchmark
args = [sys.executable, "-m", "imse_benchmark", STAGE, "--dataset", DATASET,
        "--work-dir", str(WORK), "--device", DEVICE]
if BACKBONE:
    args += ["--backbone", BACKBONE]
subprocess.run(args, cwd=REPO, check=True)